# Harvest Slot DL 학습 파이프라인

이 노트북은 **노트북 파일 1개 + Kaggle에 연결한 데이터셋**만으로 실행되도록 구성한 self-contained 학습 문서입니다.

외부 `ai/dl/freshness/*.py` 파일을 import하지 않습니다. 데이터 처리, Dataset/DataLoader, ResNet18 모델, 학습 루프, checkpoint 선정, 추론, FastAPI 응답 변환에 필요한 코드를 모두 노트북 안에 포함합니다.


## 전체 흐름

1. Kaggle/로컬 실행 환경과 GPU 확인
2. 데이터 위치 자동 탐색
3. QC 원천 데이터 또는 준비된 ImageFolder 데이터 처리
4. 데이터 기준과 라벨 매핑 설명
5. 보조 이미지 특징과 신선도 점수 rule 정의
6. Dataset/DataLoader 구성
7. ResNet18 학습 및 best checkpoint 저장
8. 최종 모델 선정 기준 확인
9. 단일 이미지 추론
10. FastAPI 연동용 응답 변환 준비


In [ ]:
# 1단계: 실행 환경, 공통 라이브러리, GPU 상태를 확인합니다.
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import copy
import json
import random
import re
import shutil
from typing import Any

import numpy as np
from PIL import Image

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, f1_score

IS_KAGGLE = Path("/kaggle").exists()
NOTE_DIR = Path.cwd()
PROJECT_ROOT = NOTE_DIR.parent if NOTE_DIR.name == "Note" else NOTE_DIR
WORKING_ROOT = Path("/kaggle/working") if IS_KAGGLE else PROJECT_ROOT / "ai" / "dl" / "freshness"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

print("IS_KAGGLE:", IS_KAGGLE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORKING_ROOT:", WORKING_ROOT)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 1. 기본 설정

현재 MVP 대상은 `apple/fuji`입니다. QC 원천 데이터의 `L/M/S` 등급을 서비스에서 사용할 `A/B/C` 등급으로 바꿔 학습합니다.

Kaggle에서는 `/kaggle/input`이 읽기 전용이므로, 새로 split한 데이터와 모델 checkpoint는 `/kaggle/working` 아래에 저장합니다.


In [ ]:
# 2단계: 과일, 라벨, 학습 경로, 모델 저장 경로를 정의합니다.
TARGET_FRUIT = "apple"
TARGET_VARIETY = "fuji"
LABELS = ["A", "B", "C"]
QC_LABEL_MAP = {"L": "A", "M": "B", "S": "C"}
QC_GRADE_MAP = {"A": "특", "B": "상", "C": "보통"}
DECISIONS = ["PASS", "REVIEW", "HOLD"]

DEFAULT_IMAGE_SIZE = 224
DEFAULT_MODEL_NAME = "resnet18"
DEFAULT_MODEL_VERSION = "freshness-resnet18-v0"

TRAIN_RATIO = 0.7
VALID_RATIO = 0.1
TEST_RATIO = 0.2
RANDOM_SEED = 42

PREPARED_DATA_ROOT = WORKING_ROOT / "freshness-data"
MODEL_ROOT = WORKING_ROOT / "models"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("대상 과일:", TARGET_FRUIT)
print("대표 품종:", TARGET_VARIETY)
print("라벨:", LABELS)
print("QC 라벨 매핑:", QC_LABEL_MAP)
print("준비 데이터 출력 경로:", PREPARED_DATA_ROOT)
print("모델 저장 경로:", MODEL_ROOT)


## 2. 데이터 구조와 자동 탐색

이 노트북은 Kaggle에 추가한 데이터가 아래 두 형태 중 하나라고 가정합니다.

### A안: 이미 준비된 ImageFolder 구조

```text
dataset-root/
  train/apple/A
  train/apple/B
  train/apple/C
  valid/apple/A
  valid/apple/B
  valid/apple/C
  test/apple/A
  test/apple/B
  test/apple/C
```

### B안: QC 원천 폴더 구조

```text
dataset-root/
  apple_fuji/
    apple_fuji_L
    apple_fuji_M
    apple_fuji_S
```

A안이면 그대로 읽고, B안이면 노트북이 `L/M/S -> A/B/C`로 변환하면서 train/valid/test split을 만듭니다.


In [ ]:
# 3단계: Kaggle input 또는 로컬 폴더에서 데이터 루트를 자동으로 찾고, 필요하면 학습 구조로 변환합니다.
OBJECT_ID_PATTERN = re.compile(r"apple_fuji_[LMS]_(\d+)-\d+_")


def count_images_under(folder: Path) -> int:
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def is_prepared_imagefolder(root: Path, fruit_type: str = TARGET_FRUIT) -> bool:
    return all((root / split / fruit_type / label).exists() for split in ["train", "valid", "test"] for label in LABELS)


def find_prepared_data_root(search_roots: list[Path]) -> Path | None:
    for root in search_roots:
        if not root.exists():
            continue
        if is_prepared_imagefolder(root):
            return root
        for candidate in root.rglob("train"):
            parent = candidate.parent
            if is_prepared_imagefolder(parent):
                return parent
    return None


def find_qc_source_root(search_roots: list[Path]) -> Path | None:
    for root in search_roots:
        if not root.exists():
            continue
        if root.name == "apple_fuji":
            return root
        candidates = [p for p in root.rglob("apple_fuji") if p.is_dir()]
        if candidates:
            return max(candidates, key=count_images_under)
    return None


def object_id_from_name(image_path: Path) -> str:
    match = OBJECT_ID_PATTERN.search(image_path.name)
    if match:
        return match.group(1)
    return image_path.stem


def split_images_by_object(images: list[Path], rng: random.Random) -> dict[str, list[Path]]:
    groups: dict[str, list[Path]] = {}
    for image in images:
        groups.setdefault(object_id_from_name(image), []).append(image)

    group_ids = list(groups.keys())
    rng.shuffle(group_ids)
    n_total = len(group_ids)
    n_train = int(n_total * TRAIN_RATIO)
    n_valid = int(n_total * VALID_RATIO)
    if n_total >= 3:
        n_valid = max(1, n_valid)
        n_train = min(n_train, n_total - n_valid - 1)

    split_group_ids = {
        "train": group_ids[:n_train],
        "valid": group_ids[n_train : n_train + n_valid],
        "test": group_ids[n_train + n_valid :],
    }
    return {split: [image for group_id in ids for image in groups[group_id]] for split, ids in split_group_ids.items()}


def prepare_qc_source_data(source_root: Path, output_root: Path, overwrite: bool = False) -> Path:
    rng = random.Random(RANDOM_SEED)
    if overwrite and output_root.exists():
        shutil.rmtree(output_root)

    copied = {"train": 0, "valid": 0, "test": 0}
    for source_label, target_label in QC_LABEL_MAP.items():
        source_folder = source_root / f"apple_fuji_{source_label}"
        if not source_folder.exists():
            raise FileNotFoundError(f"Source label folder does not exist: {source_folder}")

        images = sorted(p for p in source_folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
        splits = split_images_by_object(images, rng)
        for split, split_images in splits.items():
            target_folder = output_root / split / TARGET_FRUIT / target_label
            target_folder.mkdir(parents=True, exist_ok=True)
            for image_path in split_images:
                target_path = target_folder / f"{source_label}_{image_path.name}"
                if target_path.exists():
                    continue
                shutil.copy2(image_path, target_path)
                copied[split] += 1

    print("Prepared QC source data.")
    print("source_root:", source_root)
    print("output_root:", output_root)
    print("copied:", copied)
    return output_root


search_roots = []
if IS_KAGGLE:
    search_roots.append(KAGGLE_INPUT_ROOT)
search_roots.extend([PROJECT_ROOT / "Data" / "Sample", PROJECT_ROOT, PREPARED_DATA_ROOT])

prepared_root = find_prepared_data_root(search_roots)
qc_source_root = find_qc_source_root(search_roots)

if prepared_root is not None:
    DATA_ROOT = prepared_root
    print("준비된 ImageFolder 데이터를 사용합니다:", DATA_ROOT)
elif qc_source_root is not None:
    DATA_ROOT = prepare_qc_source_data(qc_source_root, PREPARED_DATA_ROOT, overwrite=False)
else:
    DATA_ROOT = PREPARED_DATA_ROOT
    print("데이터를 찾지 못했습니다.")
    print("Kaggle Notebook 오른쪽 Add data에서 apple_fuji 또는 train/valid/test 구조의 데이터셋을 연결하세요.")

print("DATA_ROOT:", DATA_ROOT)


## 3. 라벨 기준과 데이터 개수 확인

모델이 예측하는 클래스는 `A/B/C`입니다. 원천 QC 데이터의 `L/M/S`는 아래 기준으로 변환합니다.

| 모델 라벨 | QC 원천 라벨 | 의미 |
|---|---|---|
| A | L | 특 |
| B | M | 상 |
| C | S | 보통 |


In [ ]:
# 4단계: split별/라벨별 이미지 개수를 확인합니다.
for split in ["train", "valid", "test"]:
    print(f"[{split}]")
    for label in LABELS:
        folder = DATA_ROOT / split / TARGET_FRUIT / label
        print(f"  {label}: {count_images(folder)} images")


## 4. 보조 이미지 특징 추출

CNN 등급 예측만 사용하지 않고, 색 선명도, 둥근 정도, 멍 의심 확률을 보조 특징으로 계산합니다.

현재 구현은 Kaggle 기본 환경에서 실행하기 쉽도록 OpenCV 없이 PIL/NumPy만 사용합니다.


In [ ]:
# 5단계: PIL/NumPy 기반 보조 특징 추출 함수를 정의합니다.
def _load_rgb(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> Image.Image:
    return Image.open(image_path).convert("RGB").resize((image_size, image_size))


def calculate_color_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> float:
    image = _load_rgb(image_path, image_size).convert("HSV")
    hsv = np.asarray(image, dtype=np.float32)
    saturation = hsv[..., 1] / 255.0
    value = hsv[..., 2] / 255.0
    score = (saturation.mean() * 0.65 + value.mean() * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_roundness_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> float:
    image = _load_rgb(image_path, image_size)
    arr = np.asarray(image, dtype=np.float32)
    border = np.concatenate(
        [arr[:8].reshape(-1, 3), arr[-8:].reshape(-1, 3), arr[:, :8].reshape(-1, 3), arr[:, -8:].reshape(-1, 3)],
        axis=0,
    )
    bg = np.median(border, axis=0)
    distance = np.linalg.norm(arr - bg, axis=2)
    mask = distance > max(18.0, float(distance.mean()))
    ys, xs = np.where(mask)
    if len(xs) < 50:
        return 50.0

    width = max(1, xs.max() - xs.min() + 1)
    height = max(1, ys.max() - ys.min() + 1)
    aspect_score = min(width, height) / max(width, height)
    fill_ratio = mask.sum() / float(width * height)
    score = (aspect_score * 0.65 + min(fill_ratio / 0.78, 1.0) * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_bruise_probability(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> float:
    image = _load_rgb(image_path, image_size)
    hsv = np.asarray(image.convert("HSV"), dtype=np.float32)
    value = hsv[..., 2] / 255.0
    saturation = hsv[..., 1] / 255.0
    dark_ratio = np.logical_and(value < 0.33, saturation > 0.18).mean()
    probability = min(1.0, dark_ratio / 0.18)
    return round(float(np.clip(probability, 0.0, 1.0)), 4)


def extract_features(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> dict[str, float]:
    return {
        "color_score": calculate_color_score(image_path, image_size),
        "roundness_score": calculate_roundness_score(image_path, image_size),
        "bruise_probability": calculate_bruise_probability(image_path, image_size),
    }


sample_images = sorted((DATA_ROOT / "train" / TARGET_FRUIT).rglob("*"))
sample_images = [p for p in sample_images if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
if sample_images:
    print("sample_image:", sample_images[0])
    print(extract_features(sample_images[0]))
else:
    print("샘플 이미지가 없어 특징 추출 예시는 건너뜁니다.")


## 5. 신선도 점수와 출고 보조 판단

CNN 예측 등급과 보조 특징을 결합해 `freshness_score`를 계산합니다.

```text
freshness_score =
  CNN 등급 점수 * 0.50
+ 색 선명도 * 0.25
+ 둥근 정도 * 0.15
+ 멍 없음 점수 * 0.10
```


In [ ]:
# 6단계: 신선도 점수 계산과 출고 보조 판단 함수를 정의합니다.
GRADE_SCORES = {
    "A": 90.0,
    "B": 70.0,
    "C": 45.0,
}


@dataclass(frozen=True)
class FreshnessResult:
    fruit_type: str
    predicted_grade: str
    freshness_score: float
    color_score: float
    roundness_score: float
    bruise_probability: float
    shipping_decision: str
    model_confidence: float
    model_version: str


def calculate_freshness_score(
    predicted_grade: str,
    color_score: float,
    roundness_score: float,
    bruise_probability: float,
) -> float:
    grade_score = GRADE_SCORES.get(predicted_grade, GRADE_SCORES["C"])
    bruise_free_score = max(0.0, min(100.0, (1.0 - bruise_probability) * 100.0))
    score = (
        grade_score * 0.50
        + color_score * 0.25
        + roundness_score * 0.15
        + bruise_free_score * 0.10
    )
    return round(max(0.0, min(100.0, score)), 2)


def make_shipping_decision(freshness_score: float, bruise_probability: float) -> str:
    if freshness_score < 60.0 or bruise_probability >= 0.5:
        return "HOLD"
    if freshness_score >= 80.0 and bruise_probability < 0.2:
        return "PASS"
    return "REVIEW"


examples = [
    {"grade": "A", "color": 88.2, "roundness": 94.5, "bruise": 0.07},
    {"grade": "B", "color": 72.0, "roundness": 80.0, "bruise": 0.18},
    {"grade": "C", "color": 55.0, "roundness": 68.0, "bruise": 0.52},
]

for item in examples:
    score = calculate_freshness_score(item["grade"], item["color"], item["roundness"], item["bruise"])
    decision = make_shipping_decision(score, item["bruise"])
    print(item, "=>", score, decision)


## 6. Dataset/DataLoader 구성

`torchvision.datasets.ImageFolder`를 사용합니다. 폴더명 `A/B/C`가 자동으로 클래스 라벨이 됩니다.


In [ ]:
# 7단계: transform, Dataset, DataLoader를 정의하고 train 데이터셋을 확인합니다.
def build_transforms(image_size: int = DEFAULT_IMAGE_SIZE, train: bool = True):
    if train:
        return transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.12),
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ]
        )

    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )


def dataset_root(split: str, fruit_type: str = TARGET_FRUIT, data_root: str | Path = DATA_ROOT) -> Path:
    return Path(data_root) / split / fruit_type


def build_dataset(
    split: str,
    fruit_type: str = TARGET_FRUIT,
    data_root: str | Path = DATA_ROOT,
    image_size: int = DEFAULT_IMAGE_SIZE,
):
    root = dataset_root(split, fruit_type, data_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset folder does not exist: {root}")
    return datasets.ImageFolder(root=str(root), transform=build_transforms(image_size, train=(split == "train")))


def build_loader(
    split: str,
    fruit_type: str = TARGET_FRUIT,
    data_root: str | Path = DATA_ROOT,
    image_size: int = DEFAULT_IMAGE_SIZE,
    batch_size: int = 16,
    num_workers: int = 0,
):
    dataset = build_dataset(split, fruit_type, data_root, image_size)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


has_train_images = any(count_images(DATA_ROOT / "train" / TARGET_FRUIT / label) > 0 for label in LABELS)
if has_train_images:
    train_dataset = build_dataset("train")
    print("classes:", train_dataset.classes)
    print("samples:", len(train_dataset))
else:
    print("아직 train 이미지가 없어 Dataset 확인을 건너뜁니다.")


## 7. ResNet18 모델 구성

MVP 모델은 ResNet18입니다. 마지막 fully-connected layer만 `A/B/C` 3개 클래스에 맞게 교체합니다.

Kaggle Internet이 꺼져 있으면 pretrained weight 다운로드가 실패할 수 있습니다. 그 경우 `USE_PRETRAINED = False`로 바꿔 실행합니다.


In [ ]:
# 8단계: ResNet18 모델 생성과 checkpoint 로드 함수를 정의합니다.
def build_model(
    model_name: str = DEFAULT_MODEL_NAME,
    num_classes: int = len(LABELS),
    pretrained: bool = False,
) -> nn.Module:
    if model_name != "resnet18":
        raise ValueError(f"Unsupported model_name: {model_name}")

    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_checkpoint(checkpoint_path: str | Path, device: torch.device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    labels = checkpoint.get("labels", list(LABELS))
    model = build_model(
        model_name=checkpoint.get("model_name", DEFAULT_MODEL_NAME),
        num_classes=len(labels),
        pretrained=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model, checkpoint


preview_model = build_model(num_classes=len(LABELS), pretrained=False)
print(type(preview_model).__name__)
print("output layer:", preview_model.fc)


## 8. 학습 설정과 실행

Kaggle GPU에서 학습하려면 Notebook 우측 설정에서 Accelerator를 GPU로 켭니다.

기본값은 바로 학습되도록 `RUN_TRAINING = True`입니다. 데이터 연결만 확인하고 싶으면 `False`로 바꾸면 됩니다. Kaggle Internet 없이도 실행되도록 pretrained weight는 기본값에서 끕니다.


In [ ]:
# 9단계: 학습 설정값을 정의합니다.
RUN_TRAINING = True
USE_PRETRAINED = False

TRAIN_CONFIG = {
    "model_name": "resnet18",
    "image_size": 224,
    "batch_size": 64,
    "epochs": 15,
    "patience": 5,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "num_workers": 2 if IS_KAGGLE else 0,
}

TRAIN_CONFIG


In [ ]:
# 10단계: 학습/검증/테스트 루프를 정의하고 best checkpoint를 저장합니다.
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / max(1, len(train_loader))


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            y_true.extend(targets.detach().cpu().tolist())
            y_pred.extend(predicted.detach().cpu().tolist())

    avg_loss = total_loss / max(1, len(data_loader))
    accuracy = accuracy_score(y_true, y_pred) if y_true else 0.0
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0
    return {"loss": avg_loss, "accuracy": accuracy, "f1": f1}


def train_model(
    data_root: Path = DATA_ROOT,
    model_root: Path = MODEL_ROOT,
    fruit_type: str = TARGET_FRUIT,
    model_name: str = DEFAULT_MODEL_NAME,
    model_version: str = DEFAULT_MODEL_VERSION,
    image_size: int = DEFAULT_IMAGE_SIZE,
    batch_size: int = 16,
    epochs: int = 10,
    patience: int = 3,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    num_workers: int = 0,
    pretrained: bool = False,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    train_loader = build_loader("train", fruit_type, data_root, image_size, batch_size, num_workers)
    valid_loader = build_loader("valid", fruit_type, data_root, image_size, batch_size, num_workers)
    test_loader = build_loader("test", fruit_type, data_root, image_size, batch_size, num_workers)

    labels = train_loader.dataset.classes
    model = build_model(model_name, num_classes=len(labels), pretrained=pretrained).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    model_root.mkdir(parents=True, exist_ok=True)
    best_f1 = -1.0
    best_model_state = None
    best_path = model_root / f"{fruit_type}_{model_name}_best.pt"
    wait = 0

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_metrics = evaluate(model, train_loader, criterion, device)
        valid_metrics = evaluate(model, valid_loader, criterion, device)
        print(
            f"epoch={epoch:03d} "
            f"loss={train_loss:.4f} train_acc={train_metrics['accuracy']:.4f} train_f1={train_metrics['f1']:.4f} "
            f"valid_loss={valid_metrics['loss']:.4f} valid_acc={valid_metrics['accuracy']:.4f} valid_f1={valid_metrics['f1']:.4f}"
        )

        if valid_metrics["f1"] > best_f1:
            best_f1 = valid_metrics["f1"]
            wait = 0
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_model_state,
                    "labels": labels,
                    "fruit_type": fruit_type,
                    "model_name": model_name,
                    "model_version": model_version,
                    "image_size": image_size,
                    "train_config": TRAIN_CONFIG,
                    "valid_metrics": valid_metrics,
                },
                best_path,
            )
            print(f"saved best checkpoint: {best_path}")
        else:
            wait += 1

        if wait >= patience:
            print("Early stopping")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_metrics = evaluate(model, test_loader, criterion, device)
    print(
        f"test_loss={test_metrics['loss']:.4f} "
        f"test_acc={test_metrics['accuracy']:.4f} test_f1={test_metrics['f1']:.4f}"
    )

    checkpoint = torch.load(best_path, map_location="cpu") if best_path.exists() else {}
    checkpoint["test_metrics"] = test_metrics
    if best_path.exists():
        torch.save(checkpoint, best_path)
    return best_path, test_metrics


BEST_CHECKPOINT_PATH = None
TEST_METRICS = None

if RUN_TRAINING and has_train_images:
    try:
        BEST_CHECKPOINT_PATH, TEST_METRICS = train_model(
            data_root=DATA_ROOT,
            model_root=MODEL_ROOT,
            image_size=TRAIN_CONFIG["image_size"],
            batch_size=TRAIN_CONFIG["batch_size"],
            epochs=TRAIN_CONFIG["epochs"],
            patience=TRAIN_CONFIG["patience"],
            lr=TRAIN_CONFIG["learning_rate"],
            weight_decay=TRAIN_CONFIG["weight_decay"],
            num_workers=TRAIN_CONFIG["num_workers"],
            pretrained=USE_PRETRAINED,
        )
    except Exception as exc:
        if USE_PRETRAINED:
            print("pretrained weight 사용 중 오류가 발생했습니다. Kaggle Internet이 꺼져 있다면 USE_PRETRAINED=False로 다시 실행하세요.")
        raise exc
else:
    print("RUN_TRAINING이 False이거나 train 이미지가 없어 학습을 건너뜁니다.")


## 9. 최종 모델 선정 기준

학습 중에는 validation macro F1이 가장 높아지는 시점의 checkpoint를 best model로 저장합니다.

최종 모델 선정 기준은 아래 순서로 봅니다.

1. validation macro F1이 가장 높은 모델
2. validation F1이 비슷하면 test macro F1과 test accuracy가 안정적인 모델
3. train 성능만 높고 valid/test 성능이 낮으면 과적합으로 판단
4. checkpoint에 labels, model_name, image_size, model_version이 포함되어 FastAPI 추론에 바로 쓸 수 있어야 함


In [ ]:
# 11단계: 저장된 best checkpoint의 메타데이터를 확인합니다.
candidate_checkpoints = [
    BEST_CHECKPOINT_PATH,
    MODEL_ROOT / "apple_resnet18_best.pt",
    WORKING_ROOT / "models" / "apple_resnet18_best.pt",
]
candidate_checkpoints = [Path(p) for p in candidate_checkpoints if p is not None]
checkpoint_path = next((p for p in candidate_checkpoints if p.exists()), None)

if checkpoint_path is None:
    print("아직 확인 가능한 checkpoint가 없습니다. 학습 완료 후 다시 실행하세요.")
else:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    print("selected_checkpoint:", checkpoint_path)
    print("model_name:", checkpoint.get("model_name"))
    print("model_version:", checkpoint.get("model_version"))
    print("labels:", checkpoint.get("labels"))
    print("image_size:", checkpoint.get("image_size"))
    print("valid_metrics:", checkpoint.get("valid_metrics"))
    print("test_metrics:", checkpoint.get("test_metrics"))


## 10. 단일 이미지 추론

최종 checkpoint와 이미지 1장을 입력하면 CNN 예측 등급, confidence, 보조 특징, 신선도 점수, 출고 보조 판단을 계산합니다.


In [ ]:
# 12단계: 단일 이미지 추론 함수를 정의하고, 샘플 이미지가 있으면 바로 실행합니다.
def predict_image(
    image_path: str | Path,
    checkpoint_path: str | Path,
    fruit_type: str = TARGET_FRUIT,
    device_name: str | None = None,
) -> FreshnessResult:
    device = torch.device(device_name or ("cuda" if torch.cuda.is_available() else "cpu"))
    model, checkpoint = load_checkpoint(checkpoint_path, device)
    labels = checkpoint["labels"]
    image_size = int(checkpoint.get("image_size", DEFAULT_IMAGE_SIZE))
    model_version = checkpoint.get("model_version", DEFAULT_MODEL_VERSION)

    transform = build_transforms(image_size=image_size, train=False)
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        probabilities = torch.softmax(model(tensor), dim=1)[0]

    confidence, index = torch.max(probabilities, dim=0)
    predicted_grade = labels[int(index.item())]
    model_confidence = round(float(confidence.item()), 4)

    features = extract_features(image_path, image_size)
    freshness_score = calculate_freshness_score(
        predicted_grade=predicted_grade,
        color_score=features["color_score"],
        roundness_score=features["roundness_score"],
        bruise_probability=features["bruise_probability"],
    )
    shipping_decision = make_shipping_decision(freshness_score, features["bruise_probability"])

    return FreshnessResult(
        fruit_type=fruit_type,
        predicted_grade=predicted_grade,
        freshness_score=freshness_score,
        color_score=features["color_score"],
        roundness_score=features["roundness_score"],
        bruise_probability=features["bruise_probability"],
        shipping_decision=shipping_decision,
        model_confidence=model_confidence,
        model_version=model_version,
    )


if checkpoint_path is not None and sample_images:
    result = predict_image(sample_images[0], checkpoint_path)
    print(asdict(result))
else:
    print("checkpoint 또는 샘플 이미지가 없어 추론 예시는 건너뜁니다.")


## 11. FastAPI 응답 스펙과 연동 준비

FastAPI에서는 `predict_image()` 결과를 아래 JSON 형태로 변환하면 됩니다. 실제 API 서버에서는 업로드 이미지 저장 경로, scan_id, image_url만 서비스 레이어에서 채우면 됩니다.


In [ ]:
# 13단계: FastAPI 응답 변환 함수를 정의합니다.
def to_api_response(result: FreshnessResult, scan_id: int, image_url: str) -> dict[str, Any]:
    return {
        "success": True,
        "data": {
            "scan_id": scan_id,
            "fruit_type": result.fruit_type,
            "predicted_grade": result.predicted_grade,
            "freshness_score": result.freshness_score,
            "color_score": result.color_score,
            "roundness_score": result.roundness_score,
            "bruise_probability": result.bruise_probability,
            "shipping_decision": result.shipping_decision,
            "model_confidence": result.model_confidence,
            "model_version": result.model_version,
            "image_url": image_url,
        },
        "message": "AI 품질 판별이 완료되었습니다. 최종 출고 여부는 점주가 확정합니다.",
    }


mock_result = FreshnessResult(
    fruit_type="apple",
    predicted_grade="A",
    freshness_score=91.3,
    color_score=88.2,
    roundness_score=94.5,
    bruise_probability=0.07,
    shipping_decision="PASS",
    model_confidence=0.86,
    model_version=DEFAULT_MODEL_VERSION,
)

to_api_response(mock_result, scan_id=501, image_url="/static/scans/scan_501.jpg")


## 최종 정리

이 노트북은 Kaggle Notebook에 **노트북 파일만 import하고, 데이터셋만 Add data로 연결하면** 실행 가능한 구조입니다.

필요한 코드는 모두 노트북 안에 포함되어 있습니다. Kaggle에서 확인할 것은 두 가지입니다.

1. Notebook Accelerator를 GPU로 설정
2. Add data로 `apple_fuji` 원천 폴더 또는 `train/valid/test/apple/A-B-C` 구조의 데이터셋 연결
